In [ ]:
import xarray as xr
import numpy as np
import os

# ============================================================
# CONFIGURACIÓN GENERAL
# ============================================================

BASE = r"C:\Users\Jhon Montano\Pictures\Correlaciones"

FILES = {
    "rice":    os.path.join(BASE, "rice_bestcluster.nc"),
    "maize":   os.path.join(BASE, "maize_bestcluster.nc"),
    "wheat":   os.path.join(BASE, "wheat_bestcluster.nc"),
    "soybean": os.path.join(BASE, "soybean_bestcluster.nc"),
}

FILE_SST = os.path.join(BASE, "SST_monthly_detrended_1981_2025_0p5deg.nc")

START_YEAR = 1982
END_YEAR   = 2016

BATCH_SIZE = 300


# ============================================================
# CARGAR Y PREPROCESAR SST UNA SOLA VEZ
# ============================================================

print("Cargando SST global...")
ds_sst = xr.open_dataset(FILE_SST)
sst = ds_sst["sst_detrended"].sel(time=slice(f"{START_YEAR}", f"{END_YEAR}"))

print("SST dims:", sst.shape)

print("\nPreprocesando SST mensual...")
sst_monthly = {}

for month in range(1, 13):
    sst_m = sst.sel(time=sst.time.dt.month == month)  # [lat, lon, time]
    
    # Convertir a [time, lat, lon]
    sst_np = np.transpose(sst_m.values, (2, 0, 1))
    
    nt, nx, ny = sst_np.shape
    
    # Aplanar -> [time, n_pixels]
    sst_flat = sst_np.reshape(nt, nx * ny)
    
    # Normalizar columnas
    sst_flat = (sst_flat - sst_flat.mean(axis=0)) / sst_flat.std(axis=0)
    
    sst_monthly[month] = sst_flat

print("✔ SST mensual listo.")


# ============================================================
# FUNCIÓN PARA PROCESAR CADA CULTIVO
# ============================================================

def procesar_cultivo(nombre, file_yield):

    print("\n===================================================")
    print(f"    PROCESANDO CULTIVO: {nombre.upper()}")
    print("===================================================\n")

    # Crear carpeta de salida
    OUT_DIR = os.path.join(BASE, f"Resultados_{nombre}")
    os.makedirs(OUT_DIR, exist_ok=True)

    # -------------------------------
    # Cargar dataset de rendimiento
    # -------------------------------
    ds_y = xr.open_dataset(file_yield)
    var = list(ds_y.data_vars.keys())[0]
    
    print(f"Variable detectada: {var}")

    y = ds_y[var].sel(time=slice(f"{START_YEAR}", f"{END_YEAR}"))

    # -------------------------------
    # Identificar pixeles agrícolas
    # -------------------------------
    mask = ~np.isnan(y.isel(time=0).values)
    lat = y.lat.values
    lon = y.lon.values

    pix_lats, pix_lons = np.where(mask)
    NPIX = len(pix_lats)

    print(f"Píxeles agrícolas detectados: {NPIX}")

    # -------------------------------
    # Función de correlación por batch
    # -------------------------------
    def correlacion_batch(indices):

        batch_lats = lat[pix_lats[indices]]
        batch_lons = lon[pix_lons[indices]]

        # y_series: shape [time=35, batch]
        Y = np.stack([
            y.sel(lat=batch_lats[k], lon=batch_lons[k]).values
            for k in range(len(indices))
        ], axis=1)

        # Normalizar por columnas
        Y = (Y - Y.mean(axis=0)) / Y.std(axis=0)

        resultados = {}

        for month in range(1, 13):
            X = sst_monthly[month]  # [35, 259200]
            R = np.dot(X.T, Y) / 35
            resultados[month] = R

        return resultados, batch_lats, batch_lons

    # -------------------------------
    # LOOP PRINCIPAL DE BATCHES
    # -------------------------------
    print("Iniciando correlación por batches...\n")

    for start in range(0, NPIX, BATCH_SIZE):
        end = min(start + BATCH_SIZE, NPIX)
        idx = np.arange(start, end)

        print(f"Batch {start} → {end} ({end-start} puntos)")

        results, blats, blons = correlacion_batch(idx)

        outfile = os.path.join(
            OUT_DIR,
            f"{nombre}_corr_batch_{start}_{end}.npz"
        )

        np.savez_compressed(
            outfile,
            correlaciones=results,
            lat=blats,
            lon=blons
        )

        print("Guardado:", outfile)

    print(f"\n✔ COMPLETADO: {nombre.upper()}")


# ============================================================
# EJECUTAR PARA TODOS LOS CULTIVOS
# ============================================================

for cult, file in FILES.items():
    procesar_cultivo(cult, file)

print("\n\n🎉 TODAS LAS CORRELACIONES PARA TODOS LOS CULTIVOS HAN SIDO GENERADAS 🎉")
